# Tool-Router Ladder: Stage A, free-running (V1-13)

The student drives real tau2 episodes on this GPU, against the frozen user simulator on OpenRouter.

**Settings:** GPU T4, Internet on. Secrets `HF_TOKEN` and `OPENROUTER_API_KEY`, both ticked.

**Quota:** about 1,000 requests per UTC day, account-wide, at 5.3 requests per episode. That is roughly 190 episodes a day. The run stops cleanly when the day's budget is spent.
- **Never run this on the same UTC day as a local `harvest.py` run.** Both share one budget, and neither machine sees the other's requests.
- Start after 7 PM CDT, when the quota resets.

**Resume:** `--hub-sync` keeps the results log and trajectories in a private HF dataset repo, so each day's session resumes where the last one stopped.

**If it prints `STOPPED EARLY`:** the provider or the network is down. Stop for the day.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN", "OPENROUTER_API_KEY"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN", "OPENROUTER_API_KEY"])

In [ ]:
!pip install -q vllm
!git clone -q https://github.com/sierra-research/tau2-bench.git /kaggle/working/tau2-bench
!cd /kaggle/working/tau2-bench && git checkout -q 672227c6b6676edc20d57ea53b7000262aae77b9 && pip install -q -e .

!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
HF_USER = "CHANGE-ME"   # your Hugging Face username
SYNC = f"{HF_USER}/tool-router-results"

### Smoke run

2 tasks, 1 seed, and a 2-turn simulator; about 5 API requests.

It checks four things:
- vLLM, tau2 and the adapter load together;
- tau2's policy and tools match the pinned snapshot;
- the harness matches the baseline;
- episodes get scored.

In [ ]:
!python src/eval_free.py --config configs/qwen05b.yaml --catalog 16 --adapter "$HF_USER/tool-router-qwen05b-sft" --smoke

### Daily run: the six Stage A cells, in order

Each command finishes its cell or stops at the day's budget, and later commands then stop immediately.

Re-run this cell each day; finished cells are skipped.

In [ ]:
for cfg, size in [("qwen05b", "0.5B"), ("qwen15b", "1.5B")]:
    for catalog in (16, 40, 80):
        print(f"===== {size} catalog {catalog} =====")
        !python src/eval_free.py --config configs/{cfg}.yaml --catalog {catalog} --adapter "$HF_USER/tool-router-{cfg}-sft" --hub-sync "$SYNC"